# Etapa 5 — Desafio Final + Avaliação no Teste (Kaggle)

Pega o melhor modelo (ResNet50) e aplica a combinação final: augmentation apropriado para histologia, scheduler Cosine, weight decay, label smoothing e early stopping. No fim, avalia no **TESTE uma única vez** e gera a matriz de confusão.

Configure: **GPU On**, **Internet On**, **Add Input** com o `pathmnist_224.npz`, e **Save Version → Save & Run All**.

💾 A última célula zipa tudo de `results/` automaticamente — é só baixar.

## 1. Setup

In [ ]:
import os, sys, glob, torch
%cd /kaggle/working
REPO_URL = 'https://github.com/SEU_USUARIO/SEU_REPO.git'  # <-- EDITE
hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
if not hit:
    !git clone $REPO_URL
    hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
ROOT = hit[0].split('/src/')[0]
os.chdir(ROOT); sys.path.insert(0, ROOT)
print('raiz:', ROOT, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip install -q medmnist scikit-learn

## 2. DataLoaders com augmentation avançado (histologia)
Treino usa flips + rotações 90° + color jitter leve; val/test sem augmentation.

In [ ]:
from src.utils.seeds import set_seed
from src.etapa2_pytorch.data_224 import make_loaders
from src.etapa5_final.augment import train_tf_advanced
set_seed(42)
DATA_ROOT = os.path.dirname(glob.glob('/kaggle/input/**/pathmnist_224.npz', recursive=True)[0])
train_loader, val_loader, test_loader = make_loaders(
    size=224, batch_size=64, num_workers=2, root=DATA_ROOT, train_transform=train_tf_advanced)
print('batches treino:', len(train_loader))

## 3. Treino final do ResNet50 (fine-tuning)
AdamW + weight decay + Cosine scheduler + label smoothing + early stopping. Salva o melhor modelo. `epochs=12` com early stopping (para antes se não melhorar).

In [ ]:
from src.etapa3_cnn_vit.models import build_model
from src.etapa5_final.train_final import train_final

CKPT = '/kaggle/working/results/checkpoints'
os.makedirs(CKPT, exist_ok=True)
best_ckpt = os.path.join(CKPT, 'etapa5_resnet50_final.pt')

model = build_model('resnet50', modo='fine_tuning', pretrained=True)
res = train_final(model, train_loader, val_loader, epochs=12, lr=1e-3,
                  weight_decay=1e-4, label_smoothing=0.1, patience=4, ckpt_path=best_ckpt)
print('melhor val_acc:', res['best_val_acc'], '| tempo(s):', res['tempo_s'], '| VRAM(MB):', res['vram_mb'])

## 4. Avaliação no TESTE (uso único!) + matriz de confusão
⚠️ Esta é a ÚNICA vez que o conjunto de teste é usado, sobre o melhor modelo.

In [ ]:
from src.etapa5_final.train_final import evaluate_test, plot_confusion
from src.utils.data import LABELS
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.load_state_dict(torch.load(best_ckpt))   # melhor epoch
y_true, y_pred, acc_teste = evaluate_test(model, test_loader, device)
print('=== ACURACIA NO TESTE (uso unico):', round(acc_teste, 4), '===')
nomes = [LABELS[str(i)][:12] for i in range(9)]
os.makedirs('results/figures', exist_ok=True)
plot_confusion(y_true, y_pred, 'results/figures/etapa5_matriz_confusao.png', class_names=nomes)

## 5. 💾 Zipar tudo para download (não perde mais nada)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/resultados', 'zip', 'results')
print('Baixe no painel Output: resultados.zip')
for r,_,fs in os.walk('results'):
    for f in fs: print(' ', os.path.join(r,f))